In [1]:
import sys

import itertools
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error
from typing import List, Tuple
from tqdm import tqdm

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs

In [2]:
csv_files = Configs.ICATER_FILES
if not csv_files:
    raise FileNotFoundError("Nenhum ficheiro encontrado em Configs.ICATER_FILES.")

df = pd.concat([pd.read_csv(f, delimiter=",") for f in csv_files], ignore_index=True)
df = df.sort_values(by=['sub', 'repID', 'movID', 'step']).reset_index(drop=True)

mapa_mov = {0: 'Pick', 1: 'Place', 6: 'Move'} 
df['movLabel'] = df['movType'].map(mapa_mov)

df["jerk"] = df.groupby(['sub', 'repID', 'movID'])['acc'].diff().fillna(0)

# Filter Wrist sensors
df = df[['sub', 'repID', 'movID', 'movType', 'step', 'xw', 'yw', 'zw', 'movLabel', 'vel', 'acc', 'jerk', 't', 'k']]
df.rename(columns={'xw': 'x', 'yw': 'y', 'zw': 'z'}, inplace=True)

sensor_columns = ['x', 'y', 'z']

df.head()

,sub,repID,movID,movType,step,x,y,z,movLabel,vel,acc,jerk,t,k
0,1,1,1,0,1,-149.384114,205.638275,897.515756,Pick,0.012553,0.029301,0.000000,0.467835,3.620442
1,1,1,1,0,2,-149.556119,205.751462,897.552819,Pick,0.012476,0.046206,0.016904,1.576076,5.642838
2,1,1,1,0,3,-149.716826,205.876359,897.590338,Pick,0.012899,0.084642,0.038437,6.110274,6.081820
3,1,1,1,0,4,-149.879233,206.024678,897.629154,Pick,0.014792,0.180363,0.095721,15.009582,4.074388
4,1,1,1,0,5,-150.076148,206.204513,897.669852,Pick,0.018866,0.354677,0.174314,28.066575,23.375669


## Filtragem

In [4]:
MAD_TO_STD_FACTOR = 1.4826  # convert MAD to standard deviation for normally distributed data

def apply_hampel_series(series: pd.Series, window_size: int = 5, n_sigmas: float = 3.0) -> np.ndarray:
    """
    Applies the Hampel filter and PCHIP interpolation to a 1D pandas Series.
    Handles boundary NaNs safely via bfill and ffill.
    """
    if series.empty:
        return series.to_numpy(dtype=np.float64)

    k = 2 * window_size + 1
    rolling_median = series.rolling(window=k, center=True).median()

    # Fast vectorized MAD calculation using rolling median of absolute deviations
    # Instead of slow Python lambda inside .apply(), we compute abs deviations via numpy/scipy
    vals = series.values
    med_vals = rolling_median.values
    
    # Calculate rolling MAD using pandas rolling median on pre-centered windows where possible,
    # or keep exact rolling apply with raw=True
    rolling_mad = series.rolling(window=k, center=True).apply(
        lambda x: np.median(np.abs(x - np.median(x))), raw=True
    )

    rolling_sigma = MAD_TO_STD_FACTOR * rolling_mad
    lower_bound = rolling_median - (n_sigmas * rolling_sigma)
    upper_bound = rolling_median + (n_sigmas * rolling_sigma)

    is_outlier = (series < lower_bound) | (series > upper_bound)
    
    if not is_outlier.any():
        return vals

    series_clean = series.copy()
    series_clean[is_outlier] = np.nan

    # Interpolate using pchip and handle boundary NaNs safely
    interpolated = series_clean.interpolate(method='pchip').bfill().ffill()
    return interpolated.values

def apply_hampel_group_matrix(raw_matrix: np.ndarray, window_size: int = 5, n_sigmas: float = 3.0) -> np.ndarray:
    """
    Applies Hampel filter across all sensor columns of a single group matrix (N_samples x N_cols).
    """
    _, n_cols = raw_matrix.shape
    clean_matrix = np.empty_like(raw_matrix, dtype=np.float64)
    
    for col_idx in range(n_cols):
        s = pd.Series(raw_matrix[:, col_idx])
        clean_matrix[:, col_idx] = apply_hampel_series(s, window_size=window_size, n_sigmas=n_sigmas)
        
    return clean_matrix

def compute_raw_jerk(raw_matrix: np.ndarray) -> np.ndarray:
    """Pre-computes the sum of absolute 3rd-order differences per column."""
    if raw_matrix.shape[0] < 4:
        return np.zeros(raw_matrix.shape[1], dtype=np.float64)
    return np.sum(np.abs(np.diff(raw_matrix, n=3, axis=0)), axis=0)

def evaluate_sg_on_cached_hampel(
    raw_groups: List[np.ndarray],
    raw_jerks: List[np.ndarray],
    hampel_groups: List[np.ndarray],
    sg_w: int,
    sg_o: int
) -> Tuple[float, float]:
    """
    Evaluates Savitzky-Golay filter across all pre-filtered Hampel matrices in 2D.
    """
    total_mse = 0.0
    total_jerk_reduction = 0.0
    valid_groups = 0

    for raw_mat, raw_jerk_vec, hampel_mat in zip(raw_groups, raw_jerks, hampel_groups):
        if raw_mat.shape[0] <= sg_w:
            continue  # Skip groups too small for the SG window

        # Vectorized Savitzky-Golay across all 3 columns simultaneously (axis=0)
        smooth_mat = savgol_filter(hampel_mat, window_length=sg_w, polyorder=sg_o, axis=0)

        # Metric 1: Signal Fidelity (Mean MSE across all columns)
        col_mses = np.mean((raw_mat - smooth_mat) ** 2, axis=0)
        group_mse = float(np.mean(col_mses))

        # Metric 2: Smoothness (Reduction in discrete Jerk across all columns)
        smooth_jerk_vec = np.sum(np.abs(np.diff(smooth_mat, n=3, axis=0)), axis=0)

        # Avoid division by zero where raw_jerk is 0
        jerk_reductions = np.where(
            raw_jerk_vec > 0,
            (raw_jerk_vec - smooth_jerk_vec) / raw_jerk_vec,
            0.0
        )
        group_jerk_red = float(np.mean(jerk_reductions))

        total_mse += group_mse
        total_jerk_reduction += group_jerk_red
        valid_groups += 1

    if valid_groups == 0:
        return float('inf'), 0.0

    return total_mse / valid_groups, total_jerk_reduction / valid_groups

## Grid Search Hampel and Savitsky Golay Filter

In [5]:
df_tuning_sample = df.copy()

hampel_windows = [3, 5, 7, 9, 11, 13]
sg_windows = np.linspace(5, 21, 9, dtype=int)
sg_orders = [2, 3, 4, 5]

# 1. Extract raw numpy arrays once per group (eliminates repeated pandas groupby overhead)
print("Extracting movement groups and pre-computing raw jerk...")
raw_groups: List[np.ndarray] = []
raw_jerks: List[np.ndarray] = []

for _, group in df_tuning_sample.groupby(['sub', 'repID', 'movID'], sort=False):
    mat = group[sensor_columns].to_numpy(dtype=np.float64)
    raw_groups.append(mat)
    raw_jerks.append(compute_raw_jerk(mat))

Extracting movement groups and pre-computing raw jerk...


In [6]:
results = []

# Valid (sg_w, sg_o) pairs where window_length is strictly greater than polyorder
valid_sg_params = [(int(sw), int(so)) for sw, so in itertools.product(sg_windows, sg_orders) if sw > so]

print("Starting Global Parameter Optimization...")
for hw in tqdm(hampel_windows, desc="Hampel Windows"):
    # Pre-compute Hampel ONLY ONCE per Hampel window size across all movements
    hampel_cached_groups = [
        apply_hampel_group_matrix(mat, window_size=hw) for mat in raw_groups
    ]

    # Evaluate all SG combinations in fast vectorized numpy
    for sw, so in valid_sg_params:
        mse, jerk_red = evaluate_sg_on_cached_hampel(
            raw_groups, raw_jerks, hampel_cached_groups, sw, so
        )

        composite_score = jerk_red / (mse + 1e-6)
        results.append({
            'hampel_w': hw,
            'sg_w': sw,
            'sg_order': so,
            'mse': mse,
            'jerk_reduction': jerk_red,
            'score': composite_score
        })

results_df = pd.DataFrame(results)

Starting Global Parameter Optimization...


Hampel Windows: 100%|██████████| 6/6 [01:09<00:00, 11.65s/it]


In [7]:
min_mse = results_df['mse'].min()
max_mse = results_df['mse'].max()
mse_range = max_mse - min_mse
results_df['fidelity_score'] = (
    1.0 - ((results_df['mse'] - min_mse) / mse_range) if mse_range > 0 else 1.0
)

min_jerk = results_df['jerk_reduction'].min()
max_jerk = results_df['jerk_reduction'].max()
jerk_range = max_jerk - min_jerk
results_df['smoothness_score'] = (
    (results_df['jerk_reduction'] - min_jerk) / jerk_range if jerk_range > 0 else 1.0
)

# Final Composite Score (70% Fidelity, 30% Smoothness)
results_df['score'] = (0.7 * results_df['fidelity_score']) + (0.3 * results_df['smoothness_score'])

# Sort and display
results_df = results_df.sort_values(by='score', ascending=False)
print("Top 3 Filter Configurations:")
print(results_df[['hampel_w', 'sg_w', 'sg_order', 'mse', 'jerk_reduction', 'score']].head(3))

best_params = results_df.iloc[0]
hampel_w = int(best_params['hampel_w']) if not pd.isna(best_params.get('hampel_w')) else 5
sg_w = int(best_params['sg_w']) if not pd.isna(best_params.get('sg_w')) else 15
sg_o = int(best_params['sg_order']) if not pd.isna(best_params.get('sg_order')) else 3

print(f"\nOptimal Parameters -> Hampel Window: {hampel_w}, SG Window: {sg_w}, SG Order: {sg_o}")

Top 3 Filter Configurations:
     hampel_w  sg_w  sg_order       mse  jerk_reduction     score
160        11    15         3  0.023754        0.124735  0.851048
195        13    15         3  0.023754        0.124735  0.851048
125         9    15         3  0.023754        0.124735  0.851048

Optimal Parameters -> Hampel Window: 11, SG Window: 15, SG Order: 3


## Apply Hampel and Savitsky-Golay Filter
In the hampel filter, I can delete the line with the outlier or replace it with the ideal local value, the problem is that if I delete the line I lose temporal continuity.

In [8]:
def apply_full_filter_group(group: pd.DataFrame) -> pd.DataFrame:
    """Applies Hampel and SG filter to all sensor columns of a group at once."""
    mat = group[sensor_columns].to_numpy(dtype=np.float64)
    if mat.shape[0] <= sg_w:
        raise ValueError(
            f"Group length ({mat.shape[0]}) must be strictly greater than SG window ({sg_w})."
        )
    hampel_mat = apply_hampel_group_matrix(mat, window_size=hampel_w)
    smooth_mat = savgol_filter(hampel_mat, window_length=sg_w, polyorder=sg_o, axis=0)
    return pd.DataFrame(smooth_mat, index=group.index, columns=smooth_cols)

smooth_cols = [f"{col}_clean_smooth" for col in sensor_columns]

In [9]:


# def apply_full_filter_group(group: pd.DataFrame) -> pd.DataFrame:
#     """Applies Hampel and SG filter to all sensor columns of a group at once."""
#     mat = group[sensor_columns].to_numpy(dtype=np.float64)
#     hampel_mat = apply_hampel_group_matrix(mat, window_size=hampel_w)

#     # Protect against movements shorter than or equal to the SG window
#     if mat.shape[0] <= sg_w:
#         return pd.DataFrame(hampel_mat, index=group.index, columns=smooth_cols)

#     smooth_mat = savgol_filter(hampel_mat, window_length=sg_w, polyorder=sg_o, axis=0)
#     return pd.DataFrame(smooth_mat, index=group.index, columns=smooth_cols)

print("Applying Hampel + SG filters to all sensor columns (x, y, z)...")

smoothed_parts = [
    apply_full_filter_group(group)
    for _, group in tqdm(df.groupby(['sub', 'repID', 'movID'], sort=False), desc="Filtering groups")
]
df[smooth_cols] = pd.concat(smoothed_parts).sort_index()

print("Done")

Applying Hampel + SG filters to all sensor columns (x, y, z)...


Filtering groups: 100%|██████████| 855/855 [00:06<00:00, 139.28it/s]

Done


### Filter and Rename Dataframe

In [10]:
kinematic_cols = ['vel', 'acc', 't', 'k', 'jerk']
metadata_cols_final = ['sub', 'repID', 'movID', 'movType', 'step', 'movLabel']

df_final = df[metadata_cols_final + smooth_cols + kinematic_cols].copy()
df_final.columns = metadata_cols_final + sensor_columns + kinematic_cols
df_final.head()

,sub,repID,movID,movType,step,movLabel,x,y,z,vel,acc,t,k,jerk
0,1,1,1,0,1,Pick,-149.538738,205.584682,897.489098,0.012553,0.029301,0.467835,3.620442,0.000000
1,1,1,1,0,2,Pick,-149.534911,205.764200,897.563175,0.012476,0.046206,1.576076,5.642838,0.016904
2,1,1,1,0,3,Pick,-149.588221,205.923111,897.614005,0.012899,0.084642,6.110274,6.081820,0.038437
3,1,1,1,0,4,Pick,-149.725492,206.074859,897.649550,0.014792,0.180363,15.009582,4.074388,0.095721
4,1,1,1,0,5,Pick,-149.973544,206.232886,897.677773,0.018866,0.354677,28.066575,23.375669,0.174314


### Save to Csv

In [11]:
cache_path = Configs.ICATER_WRIST_ONLY_FILTERED_DATA_DIR
df.to_csv(cache_path, index=False)
print(f"Processed data saved to {cache_path}")

Processed data saved to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/ICater/data/processed/wrist_only_filtered_data.csv
